In [1]:
print("🔧 Đang cài đặt thư viện timm và các gói bổ trợ...")
!pip install -q timm scikit-learn pandas matplotlib seaborn tqdm pillow torchvision torch

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

# Thiết lập Seed cố định để đảm bảo tính tái lập kết quả bài toán
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kiểm tra trạng thái GPU T4 trên Colab để đảm bảo sẵn sàng tăng tốc phần cứng
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\n" + "="*40)
print(f"🖥️  Thiết bị phần cứng Colab cấp phát: {device}")
if torch.cuda.is_available():
    print(f"🚀 Tên GPU: {torch.cuda.get_device_name(0)}")
print("="*40)

🔧 Đang cài đặt thư viện timm và các gói bổ trợ...


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\ASPIRE 7\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



🖥️  Thiết bị phần cứng Colab cấp phát: cuda
🚀 Tên GPU: NVIDIA GeForce GTX 1650


In [3]:
import os

TRAIN_DIR = r"D:\Deep-Learning-final-main\train"
VAL_DIR = r"D:\Deep-Learning-final-main\val"
TEST_DIR = r"D:\Deep-Learning-final-main\test"

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

RANDOM_STATE = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Thiết bị đang sử dụng:", device)
print("Đường dẫn tập train:", TRAIN_DIR)
print("Đường dẫn tập val:", VAL_DIR)
print("Đường dẫn tập test:", TEST_DIR)
print("Số lượng thư mục train:", len(os.listdir(TRAIN_DIR)))
print("Số lượng thư mục val:", len(os.listdir(VAL_DIR)))
print("Số lượng thư mục test:", len(os.listdir(TEST_DIR)))

Thiết bị đang sử dụng: cuda
Đường dẫn tập train: D:\Deep-Learning-final-main\train
Đường dẫn tập val: D:\Deep-Learning-final-main\val
Đường dẫn tập test: D:\Deep-Learning-final-main\test
Số lượng thư mục train: 45
Số lượng thư mục val: 45
Số lượng thư mục test: 45


In [4]:
# ==========================================
# CELL 5: DATA TRANSFORMS DEFINITION
# ==========================================
IMG_SIZE = 224 # Kích thước ảnh đầu vào tiêu chuẩn bắt buộc của mô hình ViT-Base

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Tập Validation và Test sử dụng chung một bộ tiền xử lý chuẩn hóa cố định không xoay/lật ảnh
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Đã khởi tạo cấu hình các bộ tiền xử lý hình ảnh thành công.")

✅ Đã khởi tạo cấu hình các bộ tiền xử lý hình ảnh thành công.


In [5]:
# ==========================================
# CELL 6: DATASET & DATALOADER CREATION
# ==========================================
# Sử dụng ImageFolder để tự động phát hiện các class (mèo, chó,...) dựa trên tên thư mục con
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_test_transform)

class_names = train_dataset.classes
NUM_CLASSES = len(class_names)

print(f"🏷️  Nhãn các lớp nhận diện tự động: {class_names}")
print(f"🔢 Số lượng lớp (NUM_CLASSES): {NUM_CLASSES}")
print(f"📊 Số lượng mẫu - Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Tham số tối ưu cho bộ nhớ và phần cứng GPU T4
BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

🏷️  Nhãn các lớp nhận diện tự động: ['aloevera', 'banana', 'bilimbi', 'butterfly', 'cantaloupe', 'cassava', 'cat', 'cats', 'coconut', 'corn', 'cow', 'cucumber', 'curcuma', 'dog', 'dogs', 'eggplant', 'elephant', 'galangal', 'ginger', 'guava', 'hen', 'horse', 'kale', 'lion', 'longbeans', 'mango', 'melon', 'monkey', 'orange', 'paddy', 'panda', 'papaya', 'peperchili', 'pineapple', 'pomelo', 'shallot', 'sheep', 'soybeans', 'spider', 'spinach', 'squirrel', 'sweetpotatoes', 'tobacco', 'waterapple', 'watermelon']
🔢 Số lượng lớp (NUM_CLASSES): 45
📊 Số lượng mẫu - Train: 42446 | Val: 6079 | Test: 12126


In [6]:
import timm

print("🌐 Đang tải trọng số Pre-trained của Vision Transformer (vit_tiny_patch16_224)...")
model = timm.create_model('vit_tiny_patch16_224', pretrained=True)
# Hoặc 'vit_small_patch16_224'

# Thay đổi Classifier Head (từ 1000 lớp ImageNet ban đầu thành số lớp của bạn)
model.head = nn.Linear(in_features=model.head.in_features, out_features=NUM_CLASSES)
model = model.to(device)

print(f"✅ Đã cấu hình lại Classifier Head: {model.head}")

🌐 Đang tải trọng số Pre-trained của Vision Transformer (vit_tiny_patch16_224)...


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

c:\Users\ASPIRE 7\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASPIRE 7\.cache\huggingface\hub\models--timm--vit_tiny_patch16_224.augreg_in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


✅ Đã cấu hình lại Classifier Head: Linear(in_features=192, out_features=45, bias=True)


In [7]:
# ==========================================
# CELL 8: OPTIMIZATION CONFIGURATION
# ==========================================
criterion = nn.CrossEntropyLoss()

LEARNING_RATE = 5e-5
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

EPOCHS = 10
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("⚙️  Hệ thống tối ưu cấu hình hoàn tất (AdamW + Cosine Annealing Scheduler).")

⚙️  Hệ thống tối ưu cấu hình hoàn tất (AdamW + Cosine Annealing Scheduler).


In [8]:
# ==========================================
# CELL 9: TRAIN & EVALUATION FUNCTIONS
# ==========================================
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct_preds, total_samples = 0.0, 0, 0

    for images, labels in tqdm(dataloader, desc="⚡ Training Step", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_preds += torch.sum(preds == labels.data)
        total_samples += images.size(0)

    return running_loss / total_samples, (correct_preds.float() / total_samples).item()

def evaluate_model(model, dataloader, criterion, device, desc="🔍 Evaluation Step"):
    model.eval()
    running_loss, correct_preds, total_samples = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=desc, leave=False):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_preds += torch.sum(preds == labels.data)
            total_samples += images.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total_samples, (correct_preds.float() / total_samples).item(), np.array(all_labels), np.array(all_preds)

In [9]:
# ==========================================
# CELL 10: EXECUTE TRAINING & SAVE TO DRIVE
# ==========================================
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_accuracy = 0.0

print("🚀 Tiến trình huấn luyện mạng ViT bắt đầu...")
print("=" * 65)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate_model(model, val_loader, criterion, device, desc="🔍 Validation Step")

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"📈 [Epoch {epoch+1}/{EPOCHS}]")
    print(f"   • Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"   • Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")
    print("-" * 65)

print(f"\n✅ Quá trình huấn luyện hoàn tất! Độ chính xác Validation cao nhất đạt: {best_val_accuracy*100:.2f}%")

🚀 Tiến trình huấn luyện mạng ViT bắt đầu...


⚡ Training Step:   0%|          | 0/1327 [00:18<?, ?it/s]

🔍 Validation Step:   0%|          | 0/190 [00:24<?, ?it/s]

📈 [Epoch 1/10]
   • Train Loss: 0.8500 | Train Acc: 76.62%
   • Val Loss:   0.3331 | Val Acc:   89.87%
-----------------------------------------------------------------


⚡ Training Step:   0%|          | 0/1327 [00:21<?, ?it/s]

...

🔍 Validation Step:   0%|          | 0/190 [00:24<?, ?it/s]

📈 [Epoch 9/10]
   • Train Loss: 0.0353 | Train Acc: 98.06%
   • Val Loss:   0.1645 | Val Acc:   94.23%
-----------------------------------------------------------------


⚡ Training Step:   0%|          | 0/1327 [00:23<?, ?it/s]

🔍 Validation Step:   0%|          | 0/190 [00:26<?, ?it/s]

📈 [Epoch 10/10]
   • Train Loss: 0.0327 | Train Acc: 98.10%
   • Val Loss:   0.1648 | Val Acc:   94.03%
-----------------------------------------------------------------

✅ Quá trình huấn luyện hoàn tất! Độ chính xác Validation cao nhất đạt: 0.00%
